In [ ]:
print("hsh")

## Data Parcing

In [ ]:
#!/usr/bin/env python
import logging
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")

SOURCE = Path("../data/ABB 800xA User Manual.pdf")
OUTPUT_DIR = Path("parsed-doc-advanced1")
IMAGE_RESOLUTION_SCALE = 2.0

from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import ImageRefMode
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode

# Try to pick an available OCR backend option
ocr_opts = None
try:
    from docling.datamodel.pipeline_options import RapidOcrOptions
    ocr_opts = RapidOcrOptions(force_full_page_ocr=True)
except Exception:
    try:
        from docling.datamodel.pipeline_options import EasyOcrOptions
        ocr_opts = EasyOcrOptions(force_full_page_ocr=True)
    except Exception:
        try:
            from docling.datamodel.pipeline_options import TesseractOcrOptions
            ocr_opts = TesseractOcrOptions(force_full_page_ocr=True)
        except Exception:
            print("No OCR backend available. OCR will be disabled.")
            ocr_opts = None

pipeline_options = PdfPipelineOptions(
    do_table_structure=True,
    # Disable OCR here to avoid heavy ONNX models on limited environments.
    # ocr_options=RapidOcrOptions(force_full_page_ocr=True, lang=["en"]),
    # table_structure_options=dict(
    #     do_cell_matching=False,  # Use text cells predicted from table structure model
    #     mode=TableFormerMode.ACCURATE  # Use more accurate TableFormer model
    # ),
    do_ocr=False,
    generate_page_images=True,
    generate_picture_images=True,
    generate_parsed_pages=True,
    images_scale=IMAGE_RESOLUTION_SCALE,
)

conv = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)})

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not SOURCE.exists():
    print("SOURCE PDF not found:", SOURCE)
    raise SystemExit(2)

print('Starting conversion...')
result = conv.convert(SOURCE)
md = OUTPUT_DIR / f"{SOURCE.stem}-with-images.md"
result.document.save_as_markdown(md, image_mode=ImageRefMode.EMBEDDED)
print('SAVED', md)


## Data Chunking

In [ ]:
#!/usr/bin/env python3
"""
convert_md_with_base64_images.py

Reads a Markdown file containing inline base64 images and/or image placeholders,
extracts all images, chunks the document (using docling if available, otherwise a fallback),
maps images back to chunks using placeholder-aware heuristics, writes a JSONL of chunks,
and optionally writes extracted images to disk for verification.

Edit MD_PATH and OUT_DIR at the top as needed, then run:
    python convert_md_with_base64_images.py
"""

from pathlib import Path
import re
import json
import unicodedata
import base64
from collections import deque
import sys

# -------------------------
# Configuration - edit these
# -------------------------
MD_PATH = Path("parsed-doc-advanced1") / "ABB 800xA User Manual-with-images.md"
OUT_DIR = Path("../data/processed")
STEM = "relay_manual"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / f"{STEM}.chunks.jsonl"
IMAGES_OUT_DIR = OUT_DIR / "images"
IMAGES_OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Optional: Docling / Transformers
# -------------------------
try:
    from transformers import AutoTokenizer
    from docling.chunking import HybridChunker
    from docling.document_converter import DocumentConverter
    from docling.datamodel.base_models import InputFormat

    DOCILING_AVAILABLE = True
except Exception:
    DOCILING_AVAILABLE = False

# -------------------------
# Regex patterns
# -------------------------
# Match inline base64 images: ![alt](data:image/png;base64,AAAA...)
IMAGE_PATTERN = re.compile(
    r'!\[(.*?)\]\(data:image/(?P<fmt>[A-Za-z0-9_+\-]+);base64,(?P<data>[A-Za-z0-9+/=_\-\n\r]+)\)',
    re.DOTALL,
)

# Placeholders: HTML comment <!-- image -->, literal "Image" on its own line, or inline image syntax
PLACEHOLDER_PATTERN = re.compile(
    r'<!--\s*image\s*-->|^\s*Image\s*$|!\[.*?\]\(data:image/.*?\)',
    re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

FIGURE_PATTERN = re.compile(r'Figure\s+(\d+)\.\s*(.+)', re.IGNORECASE | re.MULTILINE)
TABLE_PATTERN = re.compile(r'(Table\s+\d+\..*?(?:\n\|.*)+)', re.DOTALL | re.IGNORECASE)
BULLET_PATTERN = re.compile(r'^\s*-\s+(.*)', re.MULTILINE)

# -------------------------
# Utilities
# -------------------------
def clean(text: str) -> str:
    if not text:
        return text
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    text = re.sub(r"\n[ \t]*Page\s+\d+.*?(?=\n)", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

def fallback_chunk_text(md_text):
    """
    Simple fallback chunker: split by two or more newlines into blocks.
    Returns list of simple Chunk objects with .text and .meta.headings/.meta.doc_items.
    """
    blocks = [b.strip() for b in re.split(r"\n\s*\n", md_text) if b.strip()]

    class Chunk:
        def __init__(self, text):
            self.text = text
            self.meta = type("M", (), {})()
            self.meta.headings = []
            self.meta.doc_items = []

    return [Chunk(b) for b in blocks]

def extract_figures(text):
    figures = []
    for m in FIGURE_PATTERN.finditer(text):
        figures.append({"figure_number": int(m.group(1)), "caption": m.group(2).strip()})
    return figures

def extract_tables(text):
    tables = []
    for m in TABLE_PATTERN.finditer(text):
        block = m.group(1).strip()
        first_line = block.splitlines()[0]
        tables.append({"title": first_line, "markdown": block})
    return tables

def extract_bullets(text):
    return BULLET_PATTERN.findall(text)

# -------------------------
# Read raw markdown and extract all base64 images (in order)
# -------------------------
if not MD_PATH.exists():
    print(f"Error: MD file not found at {MD_PATH}", file=sys.stderr)
    sys.exit(1)

RAW_MD = MD_PATH.read_text(encoding="utf-8")

ALL_IMAGES = []
for i, m in enumerate(IMAGE_PATTERN.finditer(RAW_MD), start=1):
    b64 = m.group("data").replace("\n", "").replace("\r", "")
    ALL_IMAGES.append({
        "image_id": f"img_{i:05d}",
        "format": m.group("fmt"),
        "base64": b64,
        "placeholder_span": (m.start(), m.end()),
    })

print(f"Images found in markdown: {len(ALL_IMAGES)}")

PLACEHOLDERS = [m.span() for m in PLACEHOLDER_PATTERN.finditer(RAW_MD)]
print(f"Placeholders found in markdown: {len(PLACEHOLDERS)}")

# -------------------------
# Convert and chunk document (Docling if available, else fallback)
# -------------------------
chunks = []
if DOCILING_AVAILABLE:
    try:
        print("Using Docling to convert and chunk the markdown...")
        converter = DocumentConverter(allowed_formats=[InputFormat.MD])
        result = converter.convert(str(MD_PATH))
        doc = result.document

        # Try to use HybridChunker if tokenizer available
        try:
            tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
        except Exception:
            tokenizer = None

        if tokenizer is not None:
            chunker = HybridChunker(tokenizer=tokenizer, max_tokens=768, merge_peers=True)
            chunks = list(chunker.chunk(dl_doc=doc))
        else:
            exported = doc.export_to_markdown()
            chunks = fallback_chunk_text(exported)
    except Exception as e:
        print("Docling conversion failed, falling back to simple chunking. Error:", e)
        chunks = fallback_chunk_text(RAW_MD)
else:
    print("Docling not available; using fallback chunking on RAW_MD.")
    chunks = fallback_chunk_text(RAW_MD)

print("Total chunks:", len(chunks))

# -------------------------
# Mapping placeholders to images and building records
# -------------------------
all_images_deque = deque(ALL_IMAGES)  # FIFO

def chunk_contains_placeholder(chunk_text):
    return bool(PLACEHOLDER_PATTERN.search(chunk_text))

def attach_images_to_chunk_by_placeholder(chunk_text, images_deque):
    """
    Attach as many images as there are placeholders in the chunk_text (FIFO from images_deque).
    """
    imgs = []
    matches = PLACEHOLDER_PATTERN.findall(chunk_text)
    for _ in matches:
        if images_deque:
            imgs.append(images_deque.popleft())
    return imgs

def should_attach_by_heuristic(chunk_text):
    """
    Heuristic: attach an image if chunk is empty/very short or equals 'Image' or only contains an HTML comment.
    """
    t = chunk_text.strip()
    if not t:
        return True
    if t.lower() == "image":
        return True
    # If chunk is short and contains no letters beyond 'Image' placeholder
    if len(t) <= 12 and re.match(r'^[^A-Za-z0-9]*$', re.sub(r'Image', '', t, flags=re.IGNORECASE)):
        return True
    return False

records = []

for idx, chunk in enumerate(chunks):
    # chunk may be a simple object or a docling chunk; handle both
    raw_text = ""
    if hasattr(chunk, "text"):
        raw_text = clean(chunk.text or "")
    else:
        raw_text = clean(str(chunk))

    text_without_images = raw_text
    images_attached = []

    # 1) If chunk contains explicit placeholders, attach corresponding images
    if chunk_contains_placeholder(raw_text) and all_images_deque:
        images_attached.extend(attach_images_to_chunk_by_placeholder(raw_text, all_images_deque))

    # 2) If no placeholders but chunk is a short/empty block, attach next image (heuristic)
    if not images_attached and should_attach_by_heuristic(raw_text) and all_images_deque:
        images_attached.append(all_images_deque.popleft())

    # 3) If chunk text contains inline base64 (rare because docling may strip), extract them
    inline_images = []
    for m in IMAGE_PATTERN.finditer(raw_text):
        inline_images.append({
            "format": m.group("fmt"),
            "base64": m.group("data").replace("\n", "").replace("\r", ""),
        })
    if inline_images:
        images_attached.extend(inline_images)

    # Build embed_text (simple fallback: use chunk text)
    embed_text = text_without_images

    # headings and doc_items if present
    headings = []
    try:
        headings = list(getattr(chunk.meta, "headings", None) or [])
    except Exception:
        headings = []

    # element type detection (simple)
    labels = set()
    try:
        labels = {str(getattr(it, "label", "")).lower() for it in (getattr(chunk.meta, "doc_items", None) or [])}
    except Exception:
        labels = set()

    if any("table" in l for l in labels):
        etype = "table"
    elif any("formula" in l or "equation" in l for l in labels):
        etype = "formula"
    elif any("picture" in l or "figure" in l for l in labels):
        etype = "figure"
    else:
        etype = "text"

    figures = extract_figures(raw_text)
    tables = extract_tables(raw_text)
    bullets = extract_bullets(raw_text)

    record = {
        "chunk_id": f"{STEM}_c{idx+1:04d}",
        "source_pdf": f"{STEM}.pdf",
        "page": None,
        "section": headings[-1] if headings else None,
        "headings": headings,
        "element_type": etype,
        "table_flag": len(tables) > 0,
        "figure_flag": len(figures) > 0,
        "formula_flag": etype == "formula",
        "text": text_without_images,
        "embed_text": embed_text,
        "images": images_attached,
        "figures": figures,
        "tables": tables,
        "bullets": bullets,
    }
    records.append(record)

# -------------------------
# Write JSONL
# -------------------------
with OUT_PATH.open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Chunks written:", OUT_PATH)

# -------------------------
# Optionally write out attached images to files for verification
# -------------------------
written = 0
with OUT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for img in rec.get("images", []):
            iid = img.get("image_id") or f"img_auto_{written+1:05d}"
            fmt = img.get("format", "png")
            b64 = img.get("base64")
            if not b64:
                continue
            out_path = IMAGES_OUT_DIR / f"{iid}.{fmt}"
            try:
                out_path.write_bytes(base64.b64decode(b64))
                written += 1
            except Exception:
                # skip invalid base64
                pass

print(f"Wrote {written} image files to {IMAGES_OUT_DIR}")

# -------------------------
# Verification counts
# -------------------------
attached_count = 0
with OUT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        attached_count += len(rec.get("images", []))

print(f"Images extracted from RAW_MD: {len(ALL_IMAGES)}")
print(f"Images attached into JSONL: {attached_count}")
print(f"Images remaining in deque (not attached): {len(all_images_deque)}")

if attached_count != len(ALL_IMAGES):
    print("Warning: counts differ. Consider using a placeholder-offset mapping approach if many images remain unattached.")
else:
    print("Success: all images attached to chunks.")

## Data Embeddiing

In [ ]:
import base64
import io
import json
import os
import warnings
from pathlib import Path

import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer

try:
    from rapidocr_onnxruntime import RapidOCR
except Exception:
    RapidOCR = None
    warnings.warn("rapidocr_onnxruntime not available; OCR will be skipped")

IN_JSONL = Path("../data/processed/relay_manual.chunks.jsonl")
MODEL_NAME = "BAAI/bge-m3"

if not IN_JSONL.exists():
    raise SystemExit(f"Input JSONL not found: {IN_JSONL}")

print(f"Loading local embedding model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
OCR_ENGINE = RapidOCR() if RapidOCR else None


def ocr_image_base64(base64_str: str) -> str:
    if not base64_str or OCR_ENGINE is None:
        return ""
    try:
        image_bytes = base64.b64decode(base64_str)
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        results, _ = OCR_ENGINE(np.array(image))
        if not results:
            return ""
        texts = []
        for item in results:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                text = item[1]
                if isinstance(text, str) and text.strip():
                    texts.append(text.strip())
        return "\n".join(texts)
    except Exception as exc:
        print(f"OCR skipped for one image: {exc}")
        return ""


def build_embed_text(record: dict) -> str:
    parts = []

    for value in [record.get("embed_text"), record.get("text"), record.get("section")]:
        if value:
            parts.append(str(value))

    headings = record.get("headings") or []
    if headings:
        parts.append("Headings: " + " | ".join([str(h) for h in headings if h]))

    for image in record.get("images") or []:
        image_label = image.get("image_id") or "image"
        ocr_text = ocr_image_base64(image.get("base64", ""))
        if ocr_text.strip():
            parts.append(f"[OCR_IMAGE_TEXT:{image_label}]\n" + ocr_text)
            # print(f"Image {image_label}: found image text")
            # print(ocr_text)
        else:
            parts.append(f"[OCR_IMAGE_TEXT:{image_label}]\nNo text found")
    print(parts)
    return "\n\n".join(parts)


def embed_texts(texts):
    embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    return embeddings


def all_embeddings():
    with IN_JSONL.open("r", encoding="utf-8") as fh:
        records = [json.loads(line) for line in fh if line.strip()]

    print(f"Loaded {len(records)} chunk records from {IN_JSONL}")

    texts = []
    for r in records:
        texts.append(build_embed_text(r))

    embeddings = embed_texts(texts)
    print("\nEmbedding model:", MODEL_NAME)
    print("Embedding count:", len(embeddings))
    print("Embedding shape:", embeddings.shape)
    print("First embedding vector sample:", embeddings[0][:10].tolist())
    print("Done.")
    return embeddings , texts


if __name__ == "__main__":
    all_embeddings()


## Data Base connection and table Creation

In [ ]:
import psycopg2
from pgvector.psycopg2 import register_vector

HOST = "localhost"
PORT = 5434
USER = "admin"
PASSWORD = "pass123"
DATABASE = "RAG_POC"
TABLE = "rag_chunks"
SOURCE_FILE = "ABB 800xA.pdf"


def create_database():
    conn = None
    try:
        # Connect to the default 'postgres' database
        conn = psycopg2.connect(
            host=HOST,
            port=PORT,
            user=USER,
            password=PASSWORD,
            dbname="postgres"
        )
        conn.autocommit = True
        cur = conn.cursor()

        # Check if the target database exists
        cur.execute(
            "SELECT 1 FROM pg_database WHERE datname = %s",
            (DATABASE,)
        )

        if cur.fetchone() is None:
            print(f"Creating database: {DATABASE}")
            cur.execute(f'CREATE DATABASE "{DATABASE}"')
        else:
            print(f"Database already exists: {DATABASE}")

        cur.close()
        conn.close()

    except Exception as e:
        print(f"Error while creating database: {e}")


def create_table():
    conn = None
    try:
        conn = psycopg2.connect(
            host=HOST,
            port=PORT,
            user=USER,
            password=PASSWORD,
            dbname=DATABASE
        )
        register_vector(conn)
        cur = conn.cursor()

        # Enable pgvector extension inside RAG_POC
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector")

        cur.execute(f"""
            CREATE TABLE IF NOT EXISTS {TABLE} (
                chunk_id UUID PRIMARY KEY,
                source TEXT,
                page_content TEXT,
                embedding VECTOR(1024),
                metadata JSONB,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """)

        conn.commit()
        print("Table ready.")

        cur.close()
        conn.close()

    except Exception as e:
        print(f"Error while creating table: {e}")

if __name__ == "__main__":
    create_database()
    create_table()

## data Vectorization

In [ ]:

import uuid
conn = None

conn = psycopg2.connect(
    host=HOST,
    port=PORT,
    user=USER,
    password=PASSWORD,
    dbname=DATABASE
)
register_vector(conn)
cur = conn.cursor()

embeddings, texts = all_embeddings()

for text, emb in zip(texts, embeddings):

    chunk_id = str(uuid.uuid4())

    metadata = {
        "chunk_id": chunk_id,
        "source": SOURCE_FILE, 
    }

    cur.execute(
        f"""
        INSERT INTO {TABLE}
        (
            chunk_id,
            source,
            page_content,
            embedding,
            metadata
        )
        VALUES (%s,%s,%s,%s,%s)
        """,
        (
            chunk_id,
            SOURCE_FILE,
            text,
            emb,
            json.dumps(metadata)
        )
    )

conn.commit()

cur.close()
conn.close()

print("Done.")
print("Stored", len(embeddings), "embeddings.")
print("Stored", len(texts), "text.")

## DB data read

In [ ]:
import psycopg2
from pgvector.psycopg2 import register_vector

conn = psycopg2.connect(
    host=HOST,
    port=PORT,
    user=USER,
    password=PASSWORD,
    dbname=DATABASE
)

register_vector(conn)

cur = conn.cursor()

chunk_id = "cf5cd03c-cab3-4dd0-85cf-ac229061188b"

cur.execute(
    """
    SELECT page_content
    FROM rag_chunks
    WHERE chunk_id = %s;
    """,
    (chunk_id,)
)

row = cur.fetchone()

if row:
    text = row[0]
    print("Text Length:", len(text))
    print(text)
else:
    print("No record found.")

cur.close()
conn.close()